<h1 align="left">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_white.svg" width="500">
    <source media="(prefers-color-scheme: light)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
    <img alt="Logo" src="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
  </picture>
  <h2 align="left">123D: Visualization Tutorial</h1>
</h1>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from py123d.api import SceneFilter, get_filtered_scenes

## 3.1 Download Demo Logs

You can download demo logs for 123D as described in the [documentation](https://autonomousvision.github.io/py123d/installation/). After the installation and download, you can start with the tutorial.


## 3.2 Create Scenes by filtering the datasets

As in other tutorials, we first query some scenes fro visualization. 

In [ ]:
scene_filter = SceneFilter(
    datasets=["nuplan-mini"],
    split_names=None,
    duration_s=2.0,  # No duration means that the scene will include the complete log.
    history_s=1.0,
    timestamp_threshold_s=2.0,
    shuffle=True,
    map_api_required=False,  # Only include scenes/logs with an available map API.
    # pinhole_camera_ids=[PinholeCameraID.PCAM_F0],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
from py123d.datatypes import LidarID
from py123d.datatypes.time.timestamp import Timestamp

scene = scenes[0]
print(scene.dataset)

sync_timestamps = scene.get_all_iteration_timestamps()
initial_time = sync_timestamps[0].time_us

sync_timestamps_ms = (np.array([timestamp.time_us for timestamp in sync_timestamps]) - initial_time) / 1e3

timestamp_constant = Timestamp.from_s(1 / 60 + 0.005)
print(timestamp_constant.time_us)
# timestamp_constant = Timestamp.from_s(0)
pcam_timestamps = {}
for pinhole_camera_id in scene.available_pinhole_camera_ids:
    pcam_timestamps[pinhole_camera_id] = [
        timestamp.time_us for timestamp in scene.get_all_pinhole_camera_timestamps(pinhole_camera_id)
    ]
    pcam_timestamps[pinhole_camera_id] = (
        np.array(pcam_timestamps[pinhole_camera_id], dtype=np.int64) + timestamp_constant.time_us
    )
    pcam_timestamps[pinhole_camera_id] = (pcam_timestamps[pinhole_camera_id] - initial_time).astype(np.float64) / 1e3

lidar_start_timestamp = [timestamp.time_us for timestamp in scene.get_all_lidar_timestamps(LidarID.LIDAR_MERGED)]
lidar_start_timestamp = (np.array(lidar_start_timestamp) - initial_time) / 1e3

ego_timestamps = [timestamp.time_us for timestamp in scene.get_all_ego_state_se3_timestamps()]
ego_timestamps = (np.array(ego_timestamps) - initial_time) / 1e3

box_timestamps = [timestamp.time_us for timestamp in scene.get_all_box_detections_se3_timestamps()]
box_timestamps = (np.array(box_timestamps) - initial_time) / 1e3

# Sort cameras by earliest timestamp
pcam_timestamps = dict(sorted(pcam_timestamps.items(), key=lambda item: item[1][0]))

fig, ax = plt.subplots(figsize=(12, 6))
half_height = 0.35
y_offset = 3

# Explicit colors per modality
ego_color = "tab:blue"
box_color = "tab:orange"
lidar_color = "tab:green"


# Add sync timestamps as scattered thin reference lines
ax.vlines(
    sync_timestamps_ms,
    ymin=-1,
    ymax=len(pcam_timestamps) + 2.5,
    color="gray",
    alpha=1,
    linewidth=0.5,
    linestyle="--",
    label="Sync Timestamps",
)

alpha = 1.0
linewidth = 2

ax.vlines(
    ego_timestamps,
    ymin=0 - half_height,
    ymax=0 + half_height,
    label="Ego State",
    color=ego_color,
    alpha=alpha,
    linewidth=linewidth,
)

ax.vlines(
    box_timestamps,
    ymin=1 - half_height,
    ymax=1 + half_height,
    label="Box Detections",
    color=box_color,
    alpha=alpha,
    linewidth=linewidth,
)

ax.vlines(
    lidar_start_timestamp,
    ymin=2 - half_height,
    ymax=2 + half_height,
    label="Lidar Start",
    color="tab:green",
    alpha=alpha,
    linewidth=linewidth,
)


for i, (camera_id, timestamps) in enumerate(pcam_timestamps.items()):
    y = y_offset + i

    ax.vlines(
        timestamps,
        ymin=y - half_height,
        ymax=y + half_height,
        label=camera_id.name,
        color=f"C{int(camera_id) + 3 % 20}",
        alpha=alpha,
        linewidth=linewidth,
    )

ax.set_xlabel("Timestamp (ms)")
ax.set_yticks(range(len(pcam_timestamps) + 3))
ax.set_yticklabels(["Ego State", "Box Detections", "Lidar Start"] + [cam_id.name for cam_id in pcam_timestamps.keys()])
ax.set_title("Sensor Timestamps")
ax.legend(loc="upper right")
ax.set_xlim(-10, 520)

plt.tight_layout()
plt.show()

### 3.3.2 Plots of Cameras

Below, we have some plot to visualize camera observations with matplotlib. In the simples form, you can retrieve a random camera and add it to a plot:

In [ ]:
from py123d.visualization.matplotlib.camera import add_pinhole_camera_ax

iteration = 10
available_pinhole_camera_ids = scene.available_pinhole_camera_ids
if len(available_pinhole_camera_ids) > 1:
    pinhole_camera_id = np.random.choice(available_pinhole_camera_ids)  # type: ignore
    pinhole_camera = scene.get_pinhole_camera_at_iteration(iteration=iteration, camera_id=pinhole_camera_id)

    fig, ax = plt.subplots(figsize=(8, 6))
    add_pinhole_camera_ax(ax, pinhole_camera)
    ax.set_title(f"{pinhole_camera_id} at Iteration {iteration}")
    plt.show()

You can also visualize the bounding boxes in overlaid in the image with:

In [ ]:
from py123d.visualization.matplotlib.camera import add_box_detections_to_camera_ax

iteration = 0
available_pinhole_camera_ids = scene.available_pinhole_camera_ids
if len(available_pinhole_camera_ids) > 1:
    pinhole_camera_id = np.random.choice(available_pinhole_camera_ids)  # type: ignore
    pinhole_camera = scene.get_pinhole_camera_at_iteration(iteration=iteration, camera_id=pinhole_camera_id)
    box_detections = scene.get_box_detections_se3_at_iteration(iteration=iteration)
    ego_state = scene.get_ego_state_se3_at_iteration(iteration=iteration)

    fig, ax = plt.subplots(figsize=(8, 6))
    add_box_detections_to_camera_ax(ax, pinhole_camera, box_detections, ego_state)
    ax.set_title(f"{pinhole_camera_id} at Iteration {iteration}")
    plt.show()

The same applies for the Lidar point cloud. Here we add the points to the image of a random camera and Lidar scanner. The color map reflects the distance to the ego vehicle.

In [ ]:
from py123d.datatypes.sensors.lidar import LidarID
from py123d.visualization.matplotlib.camera import add_lidar_to_camera_ax

available_pinhole_camera_ids = scene.available_pinhole_camera_ids
available_lidar_ids = scene.available_lidar_ids

iteration = 0
if len(available_pinhole_camera_ids) > 1 and len(available_lidar_ids) > 0:
    pinhole_camera_id = np.random.choice(available_pinhole_camera_ids)  # type: ignore
    pinhole_camera = scene.get_pinhole_camera_at_iteration(iteration=iteration, camera_id=pinhole_camera_id)
    lidar_id = np.random.choice(available_lidar_ids)  # type: ignore
    lidar_id = LidarID.LIDAR_MERGED
    lidar = scene.get_lidar_at_iteration(iteration=iteration, lidar_id=lidar_id)

    fig, ax = plt.subplots(figsize=(8, 6))
    add_lidar_to_camera_ax(ax, pinhole_camera, lidar)
    ax.set_title(f"{pinhole_camera_id} & {lidar_id} at Iteration {iteration}")
    plt.show()

## 3.4. Viser

You can visualize a scene in 3D with our viser viewer. 
You can also run viser with:
```sh
py123d-viser scene_filter=...
```
By default, you can open viewer via: [`http://localhost:8080`](http://localhost:8080 )

Check out the [viser documentation](https://viser.studio/main/) for further information.